# U-Net (and CNN+LSTM) hyperparameter tuning with Optuna — Colab GPU

Follows our CA's advice (11/3): **Bayesian tuning** of the **U-Net**. Uses the same code as
`src/compare/` on the `kevin-crn-optuna` branch, so results line up with
`results/model_comparison/`.

**Before you start:** `Runtime → Change runtime type → T4 GPU`. Then run the cells top to bottom.

| step | what | time on a T4 (rough) |
|---|---|---|
| 1-4 | setup + data check | ~3 min |
| 5 | 2-minute plumbing test | ~2 min |
| 6 | Optuna search (25 trials) — **rerun the cell to resume if Colab disconnects** | ~1-3 h |
| 7 | retrain best settings, score on val + **test** | ~10-20 min |
| 8 | save results to Drive | seconds |

Rules the search follows (same as our comparison): trains on the 500 training clips, picks settings
using the **validation** set only, and touches the **test** set exactly once (step 7).

In [ ]:
# 1. GPU check — should print a Tesla T4 (or better). If it says "no GPU", change the runtime type.
import torch, subprocess
print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU")
assert torch.cuda.is_available(), "Runtime -> Change runtime type -> T4 GPU, then rerun"

In [ ]:
# 2. Google Drive — EDIT DRIVE_DIR to the folder that holds compare_data.zip
from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/MathWorks2B_Dataset'     # <- edit if yours is elsewhere
import os
assert os.path.exists(f'{DRIVE_DIR}/compare_data.zip'), f'compare_data.zip not found in {DRIVE_DIR}'
OPTUNA_DIR = f'{DRIVE_DIR}/optuna'        # study.db lives here so a disconnect never loses progress
os.makedirs(OPTUNA_DIR, exist_ok=True)
print('ok:', os.listdir(DRIVE_DIR))

In [ ]:
# 3. Code: clone our branch. If the repo is private, add a GitHub token as a Colab secret
#    named GITHUB_TOKEN (key icon in the left bar) and allow this notebook to use it.
BRANCH = 'kevin-crn-optuna'
REPO = 'github.com/Break-Through-Tech/MathWorks-2B-speech-background-noise-suppression-with-deep-learning.git'
try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN')
except Exception:
    token = None
url = f'https://{token}@{REPO}' if token else f'https://{REPO}'
%cd /content
!rm -rf repo && git clone -q --depth 1 --branch {BRANCH} {url} repo
%cd /content/repo
!git log --oneline -1
!pip -q install pesq pystoi optuna

In [ ]:
# 4. Data: unzip and sanity-check (500 / 50 / 50 clips, 257 x 376 spectrograms, wavs for val+test)
!unzip -q -o {DRIVE_DIR}/compare_data.zip -d /content/repo
import numpy as np, glob
for split, n in (('train', 500), ('val', 50), ('test', 50)):
    d = np.load(f'data/processed/features/{split}.npz')
    assert d['noisy_mag'].shape == (n, 257, 376) and d['clean_mag'].shape == (n, 257, 376), split
    if split != 'train':
        for kind in ('noisy', 'clean'):
            assert len(glob.glob(f'data/processed/{split}/{kind}/*.wav')) == n, (split, kind)
    print(split, d['noisy_mag'].shape, 'ok')
print('data ok')

In [ ]:
# 5. Plumbing test (~2 min): tiny data, 2 epochs, 1 trial. Scores will be bad — that's expected.
!python src/compare/tune.py --model cnn --smoke --n_trials 1 --startup 1 --study colab_smoke --storage_dir /tmp/colab_smoke
!rm -rf results/tuning/colab_smoke runs/tuning /tmp/colab_smoke

In [ ]:
# 6. Optuna search. MODEL = 'cnn' (U-Net, the CA's pick) or 'crn' (CNN+LSTM).
#    If Colab disconnects: rerun cells 1-4, then rerun THIS cell — it continues the same study.
MODEL = 'cnn'
N_TRIALS = 25
STUDY = f'{MODEL}_optuna'
import json, os
db = f'{OPTUNA_DIR}/{STUDY}'
os.makedirs(db, exist_ok=True)
done = 0
if os.path.exists(f'{db}/study.db'):
    import optuna
    done = len([t for t in optuna.load_study(study_name=STUDY, storage=f'sqlite:///{db}/study.db').trials
                if t.state.is_finished()])
remaining = max(N_TRIALS - done, 0)
print(f'{done} trials already done, running {remaining} more')
if remaining:
    !python src/compare/tune.py --model {MODEL} --study {STUDY} --n_trials {remaining} --storage_dir {db}

In [ ]:
# 6b. Look at the search: best trials first
import pandas as pd
df = pd.read_csv(f'results/tuning/{STUDY}/trials.csv')
cols = ['number', 'state', 'value'] + [c for c in df.columns if c.startswith('params_')] + \
       ['user_attrs_val_pesq_out', 'user_attrs_val_snr_gain', 'user_attrs_val_stoi_gain']
df[cols].sort_values('value', ascending=False).head(10)

In [ ]:
# 7. Retrain the best settings once (same recipe, up to 60 epochs) and score on val AND test
best = json.load(open(f'results/tuning/{STUDY}/best_params.json'))
print(json.dumps(best['params'], indent=1))
cmd = best['retrain_command']
print(cmd)
!{cmd}

In [ ]:
# 7b. Compare with the earlier runs (numbers from the repo; same val/test clips and scoring)
!python src/compare/compare.py --runs cnn cnn_floor0.1 {MODEL}_optuna_best --out optuna_{MODEL} --title "Optuna-tuned {MODEL} vs earlier runs"
print(open(f'results/model_comparison/optuna_{MODEL}.md').read())

In [ ]:
# 8. Save everything to Drive -> send the zip to Kev / add it to the branch
import shutil, time
stamp = time.strftime('%Y%m%d_%H%M')
out = f'{DRIVE_DIR}/optuna_results_{MODEL}_{stamp}'
!mkdir -p {out}
!cp -r results/tuning/{STUDY} {out}/tuning_{STUDY}
!cp -r results/model_comparison/{MODEL}_optuna_best {out}/
!cp results/model_comparison/optuna_{MODEL}* {out}/
!cp runs/compare/{MODEL}_optuna_best/best.pt {out}/{MODEL}_optuna_best.pt
print('saved to', out); print(os.listdir(out))